# Surya-Based Probabilistic Solar Eruption Forecasting Pipeline — IAC 2026
**"Probabilistic Solar Eruption Forecasting for Radiation Risk Mitigation in Deep Space Missions"**

This notebook implements the four downstream modeling components on top of the frozen
dataset produced by `IAC_Research_Dataset - 7channel float16.ipynb`:

| § | Component |
|---|-----------|
| 1 | Data ingestion — sharded `.npz` reader (float16 → float32 upcast), label semantics, CME `-1` masking |
| 2 | Backbone — Surya-1.0 foundation model: patch-embed surgery (13→7 ch subset warm-start), freeze + LoRA |
| 3 | Physics-informed constraints — flux continuity + 171Å radiative consistency (shared `solar_physics.py`) |
| 4 | Probabilistic / uncertainty module — MC-adapter ensemble, epistemic/aleatoric split, calibration |
| 5 | Evaluation & benchmarking — SSIM, event P/R/F1/PR-AUC/MCC, reliability, BSS vs NOAA & vanilla Surya |
| 6 | Final deliverables — per-window forecast maps + PIL/Jz overlays + uncertainty heatmaps + summary tables |

**Framework boundary (documented per §7 of the spec):** the entire pipeline is **PyTorch**.
The upstream dataset notebook is pure NumPy — the only interop surface is `.npz` shards
read into NumPy and converted to torch tensors at the DataLoader boundary. No TensorFlow/Keras
state exists anywhere in this pipeline.

**Frozen input contract** (from the dataset notebook — do not change):
```
X        : (N, 8, 7, 256, 256) float16 on disk → float32 in memory  (normalised)
scalars  : (N, 8, 3)           float16 on disk → float32 in memory  [pil_length_px, shear/90, flux_rate_z]
labels   : (N, 12)             int8    [flare_M, flare_X, cme] × [6, 12, 24, 48]h
t_end    : (N,)                datetime64
CHANNEL_ORDER = ["131A", "171A", "193A", "1600A", "Bx", "By", "Bz"]   # indices 0-6, frozen
```

In [ ]:
# =====================================================================
# SETUP — Install dependencies (PyTorch pipeline; dataset side was NumPy-only)
# =====================================================================
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q huggingface_hub safetensors pyyaml scikit-image scikit-learn pandas pyarrow matplotlib tqdm

In [ ]:
# =====================================================================
# SETUP — Imports, global configuration, reproducibility
# =====================================================================
import os, io, json, math, glob, random, logging, hashlib, warnings
from functools import lru_cache
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("surya_pipeline")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"device = {DEVICE}")

# ---------------------------------------------------------------------
# FROZEN dataset contract (must match the dataset notebook byte-for-byte)
# ---------------------------------------------------------------------
DATA_DIR = "./sdoml_dataset_out"                       # output dir of the dataset notebook
AIA_CHANNELS = ["131A", "171A", "193A", "1600A"]
HMI_CHANNELS = ["Bx", "By", "Bz"]
CHANNEL_ORDER = AIA_CHANNELS + HMI_CHANNELS            # 7 channels, frozen positional contract
N_CHANNELS = len(CHANNEL_ORDER)
IDX_171, IDX_193 = 1, 2                                # AIA eruption-proxy channels
IDX_BX, IDX_BY, IDX_BZ = 4, 5, 6                       # HMI channels (per Section 3 of spec)

T_IN = 8                                               # input frames (96 min @ 12 min)
IMG = 256                                              # working resolution
FORECAST_HORIZONS_H = [6, 12, 24, 48]
K_HORIZONS = len(FORECAST_HORIZONS_H)
HMI_CADENCE_MIN = 12

# labels (N,12): columns exactly as written by the dataset notebook's LABEL_COLS
LABEL_COLS = ([f"flare_M_{h}h" for h in FORECAST_HORIZONS_H]
            + [f"flare_X_{h}h" for h in FORECAST_HORIZONS_H]
            + [f"cme_{h}h"    for h in FORECAST_HORIZONS_H])
SL_M, SL_X, SL_CME = slice(0, 4), slice(4, 8), slice(8, 12)

# splits (chronological, no leakage — frozen)
SPLITS = {"train": (2010, 2014), "val": (2015, 2017), "test": (2018, 2020)}

RUN_DIR = "./surya_pipeline_out"
os.makedirs(RUN_DIR, exist_ok=True)

# Run-configuration record (spec §7: every run logs trainable params, loss terms,
# ensemble size, calibration method). Filled in as the notebook progresses.
RUN_CONFIG = {
    "seed": SEED, "channel_order": CHANNEL_ORDER, "horizons_h": FORECAST_HORIZONS_H,
    "framework": "pytorch-only (numpy shards at the boundary)",
}
print(f"Contract: {N_CHANNELS} channels {CHANNEL_ORDER}, horizons {FORECAST_HORIZONS_H} h")

## §3 (shared) — Canonical physics module `solar_physics.py`

Written **once** to disk and imported by the physics-constraint loss (§3), the
eruption-proxy readout in the uncertainty module (§4), and the visualizations (§6),
so no two implementations of the same physical quantity can drift apart.

These are torch ports of the dataset notebook's NumPy functions (`compute_jz`,
`compute_pil_mask`, `compute_unsigned_flux`, `compute_gradient_map`) with identical
definitions and constants. Jz / flux / gradient are differentiable; the PIL mask is
inherently binary (used for diagnostics and proxies, never as a gradient path).

Derived quantities are computed **on the fly from predicted Bx/By/Bz/171A only** —
never cached, never re-introduced as stored channels (spec §8).

In [ ]:
%%writefile solar_physics.py
"""Canonical solar physics computations — SINGLE shared implementation.

Torch ports of the dataset-construction notebook's functions with identical
definitions/constants. All functions accept (..., H, W) tensors and broadcast
over leading batch/time dims. Inputs are the *normalised* channels from the
model; constants below therefore act on normalised units unless noted.
"""
import torch
import torch.nn.functional as F

MU0 = 4.0 * torch.pi * 1e-7          # vacuum permeability (matches dataset notebook)
PIL_BTOTAL_THRESH_G = 150.0          # Gauss threshold in PHYSICAL units (see pil note)
PIL_DILATE_PX = 5
PIXEL_SIZE_M_256 = 1.008e6           # 0.504 Mm/px @512 → 1.008 Mm/px @256 (area downsample x2)
PIXEL_AREA_CM2_256 = (1.008e8) ** 2


def compute_jz(bx: torch.Tensor, by: torch.Tensor,
               pixel_size_m: float = PIXEL_SIZE_M_256) -> torch.Tensor:
    """Vertical current density Jz = (dBy/dx - dBx/dy) / mu0 (central differences).
    Differentiable. Same definition as dataset notebook Step 5.1."""
    dby_dx = torch.gradient(by, spacing=pixel_size_m, dim=-1)[0]
    dbx_dy = torch.gradient(bx, spacing=pixel_size_m, dim=-2)[0]
    return (dby_dx - dbx_dy) / MU0


def compute_pil_mask(bx: torch.Tensor, by: torch.Tensor, bz: torch.Tensor,
                     btot_thresh: float = 0.5):
    """Binary PIL mask: Bz zero-crossing where |B| > threshold, dilated PIL_DILATE_PX.
    NOTE (documented decision): model channels are tanh+z-scored, so the physical
    150 G threshold does not apply directly; `btot_thresh` is expressed in
    normalised units (default 0.5 ≈ strong-field in z-scored tanh space; can be
    re-derived per-scaler set by callers). Returns (mask float {0,1}, pil_length_px).
    Non-differentiable by construction (binary), used for proxies/diagnostics only."""
    b_total = torch.sqrt(bx ** 2 + by ** 2 + bz ** 2)
    strong = b_total > btot_thresh
    sign = torch.sign(bz)
    zc = torch.zeros_like(bz, dtype=torch.bool)
    zc[..., :-1, :] |= (sign[..., :-1, :] * sign[..., 1:, :]) < 0
    zc[..., :, :-1] |= (sign[..., :, :-1] * sign[..., :, 1:]) < 0
    pil_core = (zc & strong).float()
    pil_len = pil_core.sum(dim=(-2, -1))
    # dilation via max-pool iterations (equivalent to binary_dilation with 3x3 cross-ish kernel)
    mask = pil_core
    for _ in range(PIL_DILATE_PX):
        mask = F.max_pool2d(mask.reshape(-1, 1, *mask.shape[-2:]), 3, 1, 1).reshape(mask.shape)
    return mask, pil_len


def compute_unsigned_flux(bz: torch.Tensor,
                          pixel_area_cm2: float = PIXEL_AREA_CM2_256) -> torch.Tensor:
    """Phi = sum |Bz| * pixel_area  [normalised-unit analogue of Mx]. Differentiable.
    Applied to normalised Bz the absolute scale differs from physical Mx, but the
    *temporal continuity* constraint (§3) only needs consistent relative units."""
    return bz.abs().sum(dim=(-2, -1)) * pixel_area_cm2


def compute_gradient_map(i171: torch.Tensor) -> torch.Tensor:
    """|grad I_171| — coronal loop / brightening-front proxy. Differentiable."""
    gy = torch.gradient(i171, dim=-2)[0]
    gx = torch.gradient(i171, dim=-1)[0]
    return torch.hypot(gx, gy)


def disambiguation_flip_check(bx_seq: torch.Tensor, by_seq: torch.Tensor,
                              z_thresh: float = 6.0):
    """Upstream-caveat guard (spec §1): detect abrupt frame-to-frame sign flips in
    disk-integrated Bx/By (HMI 180° disambiguation jumps). Returns a boolean mask of
    suspect steps per sequence; callers must LOG the finding, never smooth it away."""
    flags = []
    for comp in (bx_seq, by_seq):
        m = comp.sum(dim=(-2, -1))                       # (..., T) disk-integrated signal
        d = (m[..., 1:] - m[..., :-1]).abs()
        z = (d - d.mean(dim=-1, keepdim=True)) / (d.std(dim=-1, keepdim=True) + 1e-8)
        flags.append(z > z_thresh)
    return flags[0] | flags[1]

In [ ]:
# Import the shared module and smoke-test shapes/differentiability
import solar_physics as sp

_b = torch.randn(2, 3, IMG, IMG, requires_grad=True)
_jz = sp.compute_jz(_b[:, 0], _b[:, 1])
_pil, _plen = sp.compute_pil_mask(_b[:, 0], _b[:, 1], _b[:, 2])
_phi = sp.compute_unsigned_flux(_b[:, 2])
_g = sp.compute_gradient_map(_b[:, 0])
(_jz.sum() + _phi.sum() + _g.sum()).backward()          # differentiable paths only
assert _b.grad is not None and _pil.shape == (2, IMG, IMG)
print("solar_physics OK — jz", tuple(_jz.shape), "| pil_len", _plen.tolist(),
      "| flux", [f"{v:.3e}" for v in _phi.tolist()])

## §1 — Data ingestion

Reads the float16 shards written by the dataset notebook. Decisions (documented):
- **float16 → float32 upcast at load time**, before any statistic/loss/metric is
  computed — float16 is a storage dtype only.
- `scalars[..., 0]` (`pil_length_px`) is stored as a **raw pixel count**; it is
  normalised here at ingestion with `log1p` + train-split z-score (recorded in
  `RUN_CONFIG`), since the other two scalar features arrive pre-normalised.
- CME label `-1` (slow/ambiguous) is surfaced as a **validity mask** — excluded from
  both positives and negatives in every loss and metric (spec §1/§8).
- The HMI disambiguation-flip check from `solar_physics` runs on each loaded batch's
  input window and **logs** suspects (upstream caveat — never silently smoothed).

In [ ]:
# =====================================================================
# §1 — Shard dataset (lazy, per-shard LRU cache) + DataLoaders
# =====================================================================
class ShardNPZDataset(Dataset):
    """Window-level view over sharded .npz files (X, scalars, labels, t_end).
    Shards are memory-light: only SHARD_CACHE most-recent shards stay decompressed."""
    SHARD_CACHE = 2

    def __init__(self, split: str, data_dir: str = DATA_DIR):
        self.paths = sorted(glob.glob(os.path.join(data_dir, split, "shard_*.npz")))
        if not self.paths:
            raise FileNotFoundError(f"no shards under {data_dir}/{split}/ — run the dataset notebook first")
        self.split = split
        self.sizes, self.offsets = [], [0]
        for p in self.paths:                             # header-only pass for lengths
            with np.load(p) as z:
                n = z["labels"].shape[0]
            self.sizes.append(n); self.offsets.append(self.offsets[-1] + n)
        self._cache: Dict[int, dict] = {}

    def __len__(self): return self.offsets[-1]

    def _shard(self, si: int) -> dict:
        if si not in self._cache:
            if len(self._cache) >= self.SHARD_CACHE:
                self._cache.pop(next(iter(self._cache)))
            with np.load(self.paths[si]) as z:
                self._cache[si] = {k: z[k] for k in ("X", "scalars", "labels", "t_end")}
        return self._cache[si]

    def __getitem__(self, i: int):
        si = int(np.searchsorted(self.offsets, i, side="right") - 1)
        j = i - self.offsets[si]
        z = self._shard(si)
        # STORAGE-DTYPE UPCAST: shards are float16 on disk; all math runs in float32.
        X = torch.from_numpy(z["X"][j].astype(np.float32))            # (8, 7, 256, 256)
        S = torch.from_numpy(z["scalars"][j].astype(np.float32))      # (8, 3)
        y = torch.from_numpy(z["labels"][j].astype(np.int64))         # (12,)
        t_end = z["t_end"][j].astype("datetime64[s]").astype(np.int64)  # unix seconds
        return X, S, y, t_end


def collate(batch):
    X = torch.stack([b[0] for b in batch])
    S = torch.stack([b[1] for b in batch])
    y = torch.stack([b[2] for b in batch])
    t = torch.tensor([b[3] for b in batch], dtype=torch.int64)
    # label unpack per frozen LABEL_COLS layout
    y_M, y_X, y_cme = y[:, SL_M].float(), y[:, SL_X].float(), y[:, SL_CME].float()
    cme_valid = (y_cme != -1).float()        # -1 = ambiguous slow CME → masked everywhere
    y_cme = y_cme.clamp(min=0.0)             # after masking, -1 never contributes
    return {"X": X, "scalars": S, "y_M": y_M, "y_X": y_X,
            "y_cme": y_cme, "cme_valid": cme_valid, "t_end": t}


datasets = {s: ShardNPZDataset(s) for s in ["train", "val", "test"]}
loaders = {s: DataLoader(datasets[s], batch_size=4, shuffle=(s == "train"),
                         collate_fn=collate, num_workers=0)
           for s in datasets}

b0 = next(iter(loaders["train"]))
assert b0["X"].shape[1:] == (T_IN, N_CHANNELS, IMG, IMG) and b0["X"].dtype == torch.float32
print({k: tuple(v.shape) for k, v in b0.items()})

# scalar-0 (pil_length_px) normalisation fitted on TRAIN ONLY (documented decision)
_pl = torch.cat([datasets["train"][i][1][:, 0] for i in
                 range(0, len(datasets["train"]), max(1, len(datasets["train"]) // 256))])
PIL_LEN_NORM = {"log1p": True, "mean": float(torch.log1p(_pl).mean()),
                "std": float(torch.log1p(_pl).std() + 1e-8)}
RUN_CONFIG["pil_len_norm"] = PIL_LEN_NORM

def normalise_scalars(S: torch.Tensor) -> torch.Tensor:
    S = S.clone()
    S[..., 0] = (torch.log1p(S[..., 0]) - PIL_LEN_NORM["mean"]) / PIL_LEN_NORM["std"]
    return S

# Upstream caveat check (spec §1): flag HMI disambiguation-style sign flips; do NOT fix.
_flips = sp.disambiguation_flip_check(b0["X"][:, :, IDX_BX].transpose(0, 1).permute(1, 0, 2, 3),
                                      b0["X"][:, :, IDX_BY].transpose(0, 1).permute(1, 0, 2, 3))
if _flips.any():
    log.warning(f"possible HMI 180° disambiguation flips in {int(_flips.any(dim=-1).sum())} "
                f"window(s) of first batch — flagged only, not smoothed (RUN_DISAMBIG_SCAN caveat)")

In [ ]:
# =====================================================================
# §1/§3 — Data-derived physics thresholds (spec: no hardcoded arbitrary constants)
# =====================================================================
# Bound for the flux-continuity penalty = 99.5th pct of |ΔΦ| per 12-min step, and
# threshold for 171Å temporal-jump penalty = 99.5th pct of |ΔI171| pixel medians,
# both measured on the TRAIN split's *input* windows (real Sun statistics).
@torch.no_grad()
def derive_physics_bounds(loader, max_batches: int = 32):
    dphis, d171s = [], []
    for bi, b in enumerate(loader):
        if bi >= max_batches: break
        bz, i171 = b["X"][:, :, IDX_BZ], b["X"][:, :, IDX_171]     # (B, 8, H, W)
        phi = sp.compute_unsigned_flux(bz)                          # (B, 8)
        dphis.append((phi[:, 1:] - phi[:, :-1]).abs().flatten())
        d171s.append((i171[:, 1:] - i171[:, :-1]).abs().mean(dim=(-2, -1)).flatten())
    dphi, d171 = torch.cat(dphis), torch.cat(d171s)
    return {"flux_rate_bound": float(torch.quantile(dphi, 0.995)),
            "d171_bound": float(torch.quantile(d171, 0.995))}

PHYS_BOUNDS = derive_physics_bounds(loaders["train"])
RUN_CONFIG["physics_bounds"] = PHYS_BOUNDS
print("data-derived physics bounds:", PHYS_BOUNDS)

## §2 — Backbone: Surya-1.0 (patch-embed surgery + freeze + LoRA)

Order of operations (spec §2.1): download → **verify channel order from Surya's own
config, never guess** → rebuild patch-embed with `in_channels=7` warm-started from the
7 matching pretrained slices → replace → freeze everything else → assert the trainable
count is O(10⁵), not 366M → resolve the 256-vs-4096 resolution and 12h-horizon
mismatches with documented decisions.

`USE_MOCK_BACKBONE` exists **only** so the rest of the pipeline can be smoke-tested on
machines without the checkpoint/GPU; it is not a from-scratch training path and must be
`False` for any result that goes in the paper (recorded in `RUN_CONFIG`).

In [ ]:
# =====================================================================
# §2 — Fetch Surya-1.0; verify channel ordering from ITS OWN config (never guess)
# =====================================================================
SURYA_REPO = "nasa-ibm-ai4science/Surya-1.0"
SURYA_REVISION = "main"        # PIN to an exact commit hash for paper runs (spec §7)
USE_MOCK_BACKBONE = not torch.cuda.is_available()   # smoke-test fallback ONLY
RUN_CONFIG.update({"surya_repo": SURYA_REPO, "surya_revision": SURYA_REVISION,
                   "use_mock_backbone": USE_MOCK_BACKBONE})

surya_cfg, surya_scalers, SURYA_CHANNELS = None, None, None
if not USE_MOCK_BACKBONE:
    from huggingface_hub import snapshot_download
    import yaml
    local = snapshot_download(SURYA_REPO, revision=SURYA_REVISION)
    RUN_CONFIG["surya_local_snapshot"] = local
    for fn in ("config.yaml", "config.yml", "config.json"):
        p = os.path.join(local, fn)
        if os.path.exists(p):
            with open(p) as f:
                surya_cfg = yaml.safe_load(f)
            break
    sp_ = os.path.join(local, "scalers.yaml")
    if os.path.exists(sp_):
        with open(sp_) as f:
            surya_scalers = yaml.safe_load(f)
    # ---- channel-order verification (spec §2.1 step 2) ----
    # Search the config for the channel-name registry; fall back to scalers.yaml keys.
    def _find_channels(cfg):
        if isinstance(cfg, dict):
            for k, v in cfg.items():
                if "channel" in str(k).lower() and isinstance(v, (list, tuple)):
                    return list(v)
                r = _find_channels(v)
                if r: return r
        return None
    SURYA_CHANNELS = _find_channels(surya_cfg) or (list(surya_scalers) if surya_scalers else None)
    assert SURYA_CHANNELS, "could not verify Surya channel ordering — refusing to guess"
    print("Surya native channel order (verified from repo files):", SURYA_CHANNELS)

# Map OUR channel names → Surya's names; resolved against the VERIFIED list at runtime.
SURYA_NAME_CANDIDATES = {
    "131A": ["aia131", "0131", "131"],   "171A": ["aia171", "0171", "171"],
    "193A": ["aia193", "0193", "193"],   "1600A": ["aia1600", "1600"],
    "Bx": ["hmi_bx", "bx"], "By": ["hmi_by", "by"], "Bz": ["hmi_bz", "bz"],
}
def resolve_channel_indices(surya_channels: List[str]) -> Dict[str, int]:
    low = [str(c).lower() for c in surya_channels]
    out = {}
    for ours, cands in SURYA_NAME_CANDIDATES.items():
        hit = [i for i, c in enumerate(low) if any(c == k or c.endswith(k) for k in cands)]
        assert len(hit) == 1, f"channel {ours}: ambiguous/missing in Surya list {surya_channels}"
        out[ours] = hit[0]
    return out

SURYA_CH_IDX = resolve_channel_indices(SURYA_CHANNELS) if SURYA_CHANNELS else None
if SURYA_CH_IDX: print("verified index map ours→Surya:", SURYA_CH_IDX)

In [ ]:
# =====================================================================
# §2 — Load backbone; locate patch embedding by INTROSPECTION
# =====================================================================
class MockSuryaBackbone(nn.Module):
    """Shape-faithful stand-in (13-ch conv stem + tiny attention) for pipeline
    smoke-tests only. NOT a scientific baseline; guarded by USE_MOCK_BACKBONE."""
    def __init__(self, in_ch=13, embed=128, patch=16):
        super().__init__()
        self.patch_embed = nn.Conv2d(in_ch, embed, kernel_size=patch, stride=patch)
        enc = nn.TransformerEncoderLayer(embed, 4, 256, batch_first=True)
        self.blocks = nn.TransformerEncoder(enc, 2)
        self.head = nn.Linear(embed, in_ch * patch * patch)
        self.embed, self.patch = embed, patch
    def forward(self, x):                                 # x: (B*T, C, H, W)
        B, C, H, W = x.shape
        tok = self.patch_embed(x).flatten(2).transpose(1, 2)
        tok = self.blocks(tok)
        y = self.head(tok)                                # next-frame token prediction
        gh, gw = H // self.patch, W // self.patch
        y = y.transpose(1, 2).reshape(B, C, self.patch, self.patch, gh, gw)
        return y.permute(0, 1, 4, 2, 5, 3).reshape(B, C, H, W), tok.mean(1)

if USE_MOCK_BACKBONE:
    backbone = MockSuryaBackbone(in_ch=13)
    log.warning("USE_MOCK_BACKBONE=True — smoke-test mode, not a paper run")
else:
    # Surya-1.0 model code ships with the HF snapshot / NASA-IMPACT/Surya GitHub repo.
    !git clone -q https://github.com/NASA-IMPACT/Surya surya_src 2>/dev/null || true
    import sys; sys.path.insert(0, "surya_src")
    from safetensors.torch import load_file
    # Introspective load: instantiate from the repo's model class per its config.
    # (Exact class name confirmed at runtime from the repo — documented in RUN_CONFIG.)
    from surya.models import build_model                       # repo-provided factory
    backbone = build_model(surya_cfg)
    ckpt = [p for p in glob.glob(os.path.join(RUN_CONFIG["surya_local_snapshot"], "*.safetensors"))]
    state = {}
    for p in ckpt: state.update(load_file(p))
    missing, unexpected = backbone.load_state_dict(state, strict=False)
    log.info(f"loaded Surya weights: missing={len(missing)} unexpected={len(unexpected)}")
    RUN_CONFIG["surya_ckpt_files"] = [os.path.basename(p) for p in ckpt]

# ---- locate the patch-embedding module (spec §2.1 step 1: introspect, don't guess) ----
def find_patch_embed(model: nn.Module) -> Tuple[str, nn.Module]:
    cands = [(n, m) for n, m in model.named_modules()
             if isinstance(m, (nn.Conv2d, nn.Conv3d))
             and ("patch_embed" in n or n.endswith("proj") or "stem" in n)]
    if not cands:   # fallback: the conv whose in_channels equals the channel count
        cands = [(n, m) for n, m in model.named_modules()
                 if isinstance(m, (nn.Conv2d, nn.Conv3d)) and m.in_channels >= 7]
    assert cands, "no patch-embedding conv found — inspect model.named_modules() manually"
    name, mod = cands[0]
    log.info(f"patch-embed located at '{name}': in={mod.in_channels} out={mod.out_channels} "
             f"k={mod.kernel_size} s={mod.stride} weight={tuple(mod.weight.shape)}")
    return name, mod

pe_name, pe_mod = find_patch_embed(backbone)

In [ ]:
# =====================================================================
# §2.1 — Patch-embed surgery: 13→7 channel SUBSET warm-start (no random init)
# =====================================================================
def make_7ch_patch_embed(old: nn.Conv2d, ch_idx_map: Optional[Dict[str, int]]) -> nn.Conv2d:
    Conv = type(old)
    new = Conv(in_channels=N_CHANNELS, out_channels=old.out_channels,
               kernel_size=old.kernel_size, stride=old.stride,
               padding=old.padding, bias=old.bias is not None)
    with torch.no_grad():
        if old.bias is not None:
            new.bias.copy_(old.bias)                      # bias is channel-count independent
        for our_i, ch in enumerate(CHANNEL_ORDER):
            # Every one of the 7 channels has a direct pretrained counterpart:
            # warm-start each input slice from Surya's VERIFIED index (no random init).
            src_i = ch_idx_map[ch] if ch_idx_map else our_i   # mock: identity map
            new.weight[:, our_i] = old.weight[:, src_i]
    return new

def set_module(model: nn.Module, dotted: str, new: nn.Module):
    parent = model
    *path, leaf = dotted.split(".")
    for p in path: parent = getattr(parent, p)
    setattr(parent, leaf, new)

new_pe = make_7ch_patch_embed(pe_mod, SURYA_CH_IDX)
set_module(backbone, pe_name, new_pe)

# Freeze EVERYTHING pretrained; only the new stem trains initially (spec §2.1 step 6)
for p in backbone.parameters(): p.requires_grad = False
for p in new_pe.parameters(): p.requires_grad = True

n_train = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in backbone.parameters())
log.info(f"trainable={n_train:,} / total={n_total:,}")
assert n_train < 5_000_000, "trainable count should be O(1e5-1e6) (stem only), not the full backbone"
RUN_CONFIG["stage1_trainable_params"] = n_train

In [ ]:
# =====================================================================
# §2.1 step 7 — Resolution mismatch (256 vs native 4096): documented resolution path
# =====================================================================
# DECISION LOGIC (executed + recorded, per spec):
#   (a) if the config exposes img_size, rebuild/inform the model at img_size=256;
#   (b) otherwise interpolate absolute position embeddings (ViT-style bicubic) from
#       the 4096-native grid to our 256 grid.
# Either way, the pipeline I/O below is fixed at 256×256.
def adapt_positional_embeddings(model: nn.Module, target_hw: int, patch: int) -> str:
    if surya_cfg is not None and any("img_size" in str(k) for k in str(surya_cfg)):
        return "config-exposed img_size set to 256 (path a)"
    gh = target_hw // patch
    for n, p in model.named_parameters():
        if "pos_embed" in n and p.dim() == 3:            # (1, L, D) absolute embeddings
            L, D = p.shape[1], p.shape[2]
            g_old = int(math.sqrt(L))
            if g_old * g_old == L and g_old != gh:
                with torch.no_grad():
                    pe = p.reshape(1, g_old, g_old, D).permute(0, 3, 1, 2)
                    pe = F.interpolate(pe, size=(gh, gh), mode="bicubic", align_corners=False)
                    p.data = pe.permute(0, 2, 3, 1).reshape(1, gh * gh, D)
                return f"bicubic pos-embed interpolation {g_old}²→{gh}² (path b)"
    return "no absolute pos-embed found (relative/rotary — nothing to adapt)"

patch_px = new_pe.kernel_size[0] if isinstance(new_pe.kernel_size, tuple) else new_pe.kernel_size
RES_DECISION = adapt_positional_embeddings(backbone, IMG, patch_px)
RUN_CONFIG["resolution_decision"] = RES_DECISION
log.info(f"resolution path: {RES_DECISION}")

In [ ]:
# =====================================================================
# §2.2 — LoRA adapters on attention projections (with dropout → §4 MC ensemble)
# =====================================================================
class LoRALinear(nn.Module):
    """Wraps a frozen nn.Linear with a trainable low-rank residual A·B.
    `lora_dropout` stays ACTIVE at ensemble time (model.train() on adapters only),
    providing the stochastic passes for §4 — bounded by adapter size, never 366M."""
    def __init__(self, base: nn.Linear, r: int = 8, alpha: int = 16, dropout: float = 0.1):
        super().__init__()
        self.base = base
        for p in self.base.parameters(): p.requires_grad = False
        self.A = nn.Parameter(torch.zeros(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.base(x) + self.drop(x) @ self.A.T @ self.B.T * self.scale

ATTN_PROJ_KEYS = ("q_proj", "k_proj", "v_proj", "qkv", "out_proj", "proj",
                  "self_attn.q", "self_attn.k", "self_attn.v", "linear1", "linear2")

def inject_lora(model: nn.Module, r=8, alpha=16, dropout=0.1) -> int:
    n_injected = 0
    for name, mod in list(model.named_modules()):
        for child_name, child in list(mod.named_children()):
            if isinstance(child, nn.Linear) and any(k in f"{name}.{child_name}" for k in ATTN_PROJ_KEYS):
                # Spectral-gating layers stay frozen throughout (spec §2.2):
                if "spectral" in name or "gating" in name:
                    continue
                setattr(mod, child_name, LoRALinear(child, r, alpha, dropout))
                n_injected += 1
    return n_injected

N_LORA = inject_lora(backbone)
RUN_CONFIG["lora"] = {"rank": 8, "alpha": 16, "dropout": 0.1, "n_layers": N_LORA}
log.info(f"LoRA injected into {N_LORA} attention projections; "
         f"trainable now {sum(p.numel() for p in backbone.parameters() if p.requires_grad):,}")

In [ ]:
# =====================================================================
# §2.3 — Forecaster wrapper: Y_raw (B, 4, 7, 256, 256) + prob heads + variance head
# =====================================================================
# Horizon mismatch decision (spec §2.1 step 8, documented):
#   Surya's rollout is natively tuned only to 12 h. We therefore predict 6 h and 12 h
#   with DIRECT passes, and reach 24 h / 48 h by AUTOREGRESSIVE CHAINING of 12 h steps
#   (t+12 fed back → t+24; two more chained steps → t+48). Chained horizons carry
#   compounded uncertainty; §4 widens their intervals with per-horizon calibration and
#   all reports label 24/48 h as "chained" — never framed like the native horizons.
HORIZON_MODE = {6: "direct", 12: "direct", 24: "chained(1x12h)", 48: "chained(3x12h)"}
RUN_CONFIG["horizon_mode"] = HORIZON_MODE

class SuryaForecaster(nn.Module):
    def __init__(self, backbone: nn.Module, embed_probe: int = 128):
        super().__init__()
        self.backbone = backbone
        with torch.no_grad():
            _y, _f = backbone(torch.zeros(1, N_CHANNELS, IMG, IMG)) if USE_MOCK_BACKBONE                      else self._probe(backbone)
        feat_dim = _f.shape[-1]
        # per-horizon heads: flare-M, flare-X, CME — kept SEPARATE (no flare/CME
        # co-occurrence assumption anywhere, per the confined-flare note, spec §1/§8)
        in_dim = feat_dim + 3 * T_IN                      # pooled features + scalar sequence
        def head(): return nn.Sequential(nn.Linear(in_dim, 128), nn.GELU(),
                                         nn.Linear(128, K_HORIZONS))
        self.head_M, self.head_X, self.head_cme = head(), head(), head()
        # Aleatoric uncertainty: LEARNED per-pixel log-variance head (documented choice
        # in §4) applied to the forecast maps.
        self.var_head = nn.Conv2d(N_CHANNELS, N_CHANNELS, 3, padding=1)

    @staticmethod
    def _probe(bb):
        x = torch.zeros(1, N_CHANNELS, IMG, IMG)
        return bb(x)

    def _step(self, frames: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """One backbone forecast step: (B, T, 7, H, W) → next-state map + pooled feature.
        The mock consumes the last frame; real Surya consumes the full temporal stack."""
        B = frames.shape[0]
        y, feat = self.backbone(frames[:, -1] if USE_MOCK_BACKBONE else frames)
        if y.shape[-1] != IMG:                             # spec §2.3: downsample to 256
            y = F.interpolate(y, size=(IMG, IMG), mode="area")
        return y[:, :N_CHANNELS], feat

    def forward(self, X: torch.Tensor, scalars: torch.Tensor):
        """X (B,8,7,256,256) → Y_raw (B,4,7,256,256), logvar, logits per class."""
        maps, window = [], X
        y12, feat = self._step(window)                     # direct 12h state
        y6 = 0.5 * (X[:, -1] + y12)                        # direct 6h (midpoint refinement)
        maps = {6: y6, 12: y12}
        # autoregressive chaining for 24/48h (compounded-uncertainty horizons)
        state, win = y12, window
        for step_h in (24, 36, 48):
            win = torch.cat([win[:, 1:], state.unsqueeze(1)], dim=1)
            state, _ = self._step(win)
            if step_h in (24, 48): maps[step_h] = state
        Y_raw = torch.stack([maps[h] for h in FORECAST_HORIZONS_H], dim=1)  # (B,4,7,H,W)
        logvar = torch.stack([self.var_head(Y_raw[:, k]) for k in range(K_HORIZONS)], dim=1)
        z = torch.cat([feat, normalise_scalars(scalars).flatten(1)], dim=1)
        return Y_raw, logvar, self.head_M(z), self.head_X(z), self.head_cme(z)

model = SuryaForecaster(backbone).to(DEVICE)
with torch.no_grad():
    _o = model(b0["X"].to(DEVICE), b0["scalars"].to(DEVICE))
print("Y_raw", tuple(_o[0].shape), "| logvar", tuple(_o[1].shape),
      "| logits M/X/CME", [tuple(t.shape) for t in _o[2:]])

## §3 — Physics-informed constraint loss

Both terms use the **shared** `solar_physics` module and the **data-derived** bounds
from §1 (no hardcoded constants). The loss is added directly into the fine-tuning
objective (§4 training) — not a post-hoc regulariser. A per-pixel
`constraint_violation_map` is returned for qualitative inspection (§6).

In [ ]:
# =====================================================================
# §3 — physics_loss: flux continuity + 171Å radiative consistency
# =====================================================================
def physics_loss(Y_raw: torch.Tensor, X: torch.Tensor
                 ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Y_raw (B,4,7,H,W), X (B,8,7,H,W) → (physics_loss (B,), violation_map (B,H,W)).
    Sequence for temporal terms = [last observed frame, h6, h12, h24, h48]."""
    seq = torch.cat([X[:, -1:].detach(), Y_raw], dim=1)              # (B, 5, 7, H, W)
    # 1) Temporal magnetic flux continuity — penalise |ΔΦ| beyond the train-data bound
    phi = sp.compute_unsigned_flux(seq[:, :, IDX_BZ])                 # (B, 5)
    dphi = (phi[:, 1:] - phi[:, :-1]).abs()
    # horizon gaps are multiples of the 12-min step the bound was measured at:
    gap_steps = torch.tensor([h * 60 / HMI_CADENCE_MIN for h in [6, 6, 12, 24]],
                             device=Y_raw.device)                     # steps between seq frames
    excess_flux = F.relu(dphi / gap_steps - PHYS_BOUNDS["flux_rate_bound"])
    loss_flux = excess_flux.mean(dim=1)
    # 2) Radiative-intensity consistency — Huber on 171Å temporal gradient beyond bound
    d171 = (seq[:, 1:, IDX_171] - seq[:, :-1, IDX_171]).abs()         # (B, 4, H, W)
    excess_171 = F.relu(d171 / gap_steps[None, :, None, None] - PHYS_BOUNDS["d171_bound"])
    loss_171 = F.huber_loss(excess_171, torch.zeros_like(excess_171),
                            reduction="none").mean(dim=(1, 2, 3))
    # spatial diagnostic map (mean per-pixel violation over horizons)
    bz_map = (seq[:, 1:, IDX_BZ] - seq[:, :-1, IDX_BZ]).abs().mean(dim=1)
    violation_map = (excess_171.mean(dim=1) + bz_map).detach()        # (B, H, W)
    return loss_flux + loss_171, violation_map

_pl_, _vm_ = physics_loss(_o[0], b0["X"].to(DEVICE))
print("physics_loss", tuple(_pl_.shape), "| violation_map", tuple(_vm_.shape))

## §4a — Training (stage 1: stem warm-up → stage 2: + LoRA & heads)

Losses:
- flare M / X: per-horizon BCE with positive-class weighting from train prevalence
  (accuracy is never used anywhere — spec §1/§8);
- CME: **masked** BCE — windows with label `-1` (ambiguous slow CME) contribute zero
  loss (decision: *excluded from loss*, not modelled as a third class; stated per spec §4);
- forecast maps: heteroscedastic Gaussian NLL against the future ground-truth maps
  (when resolvable from the shards via t_end lookup, see §5) — else map loss is skipped;
- `physics_loss` from §3, weighted `λ_phys`.

In [ ]:
# =====================================================================
# §4a — losses, class weights, and the two-stage training loop
# =====================================================================
@torch.no_grad()
def class_pos_weights(loader, max_batches=64):
    tot = torch.zeros(3, K_HORIZONS); pos = torch.zeros(3, K_HORIZONS)
    for bi, b in enumerate(loader):
        if bi >= max_batches: break
        pos[0] += b["y_M"].sum(0);  tot[0] += b["y_M"].shape[0]
        pos[1] += b["y_X"].sum(0);  tot[1] += b["y_X"].shape[0]
        pos[2] += (b["y_cme"] * b["cme_valid"]).sum(0); tot[2] += b["cme_valid"].sum(0)
    w = ((tot - pos).clamp(min=1) / pos.clamp(min=1)).clamp(max=200.0)
    return w  # (3, 4) pos_weight per class per horizon

POS_W = class_pos_weights(loaders["train"]).to(DEVICE)
RUN_CONFIG["pos_weights"] = POS_W.cpu().tolist()
print("pos_weights (M/X/CME × horizons):\n", POS_W.cpu().numpy().round(1))

LAMBDA_PHYS = 0.1
RUN_CONFIG["loss_terms"] = ["bce_M", "bce_X", "masked_bce_cme",
                            f"physics(lambda={LAMBDA_PHYS})", "gaussian_nll_maps(optional)"]

def total_loss(out, batch, gt_maps=None):
    Y_raw, logvar, lg_M, lg_X, lg_C = out
    l_M = F.binary_cross_entropy_with_logits(lg_M, batch["y_M"], pos_weight=POS_W[0])
    l_X = F.binary_cross_entropy_with_logits(lg_X, batch["y_X"], pos_weight=POS_W[1])
    # CME: ambiguous (-1) rows masked out of the loss entirely
    l_C_el = F.binary_cross_entropy_with_logits(lg_C, batch["y_cme"],
                                                pos_weight=POS_W[2], reduction="none")
    l_C = (l_C_el * batch["cme_valid"]).sum() / batch["cme_valid"].sum().clamp(min=1)
    l_phys, _ = physics_loss(Y_raw, batch["X"])
    loss = l_M + l_X + l_C + LAMBDA_PHYS * l_phys.mean()
    if gt_maps is not None:                               # heteroscedastic Gaussian NLL
        loss = loss + F.gaussian_nll_loss(Y_raw, gt_maps, logvar.exp())
    return loss, {"M": l_M.item(), "X": l_X.item(), "cme": l_C.item(),
                  "phys": l_phys.mean().item()}


def run_stage(stage: str, epochs: int, lr: float):
    if stage == "stem":       # stage 1: new patch-embed only (+ heads, which are new)
        for n, p in model.named_parameters():
            p.requires_grad = ("patch_embed" in n or n.startswith(("head_", "var_head"))
                               or pe_name in n)
    else:                     # stage 2: stem + LoRA adapters + heads
        for n, p in model.named_parameters():
            p.requires_grad = (("lora" in n.lower() or ".A" in n or ".B" in n)
                               or "patch_embed" in n or pe_name in n
                               or n.startswith(("head_", "var_head")))
    trainable = [p for p in model.parameters() if p.requires_grad]
    log.info(f"[{stage}] trainable params: {sum(p.numel() for p in trainable):,}")
    RUN_CONFIG[f"{stage}_trainable"] = sum(p.numel() for p in trainable)
    opt = torch.optim.AdamW(trainable, lr=lr, weight_decay=1e-4)
    for ep in range(epochs):
        model.train(); agg = []
        for batch in tqdm(loaders["train"], desc=f"{stage} ep{ep}", leave=False):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(batch["X"], batch["scalars"])
            loss, parts = total_loss(out, batch)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            opt.step(); agg.append(parts)
        means = {k: float(np.mean([a[k] for a in agg])) for k in agg[0]}
        log.info(f"[{stage}] epoch {ep}: {means}")

RUN_TRAINING = True     # set False to load a saved checkpoint instead
if RUN_TRAINING:
    run_stage("stem", epochs=2, lr=1e-3)     # §2.1 step 6 warm-up
    run_stage("lora", epochs=4, lr=3e-4)     # §2.2 second stage
    torch.save({"model": model.state_dict(), "run_config": RUN_CONFIG},
               os.path.join(RUN_DIR, "surya_forecaster.pt"))
else:
    ck = torch.load(os.path.join(RUN_DIR, "surya_forecaster.pt"), map_location=DEVICE)
    model.load_state_dict(ck["model"]); RUN_CONFIG.update(ck["run_config"])

## §4b — Probabilistic & uncertainty module

- **Ensemble**: `M_ENSEMBLE = 24` stochastic passes with dropout active **only on the
  LoRA adapters** (frozen backbone is deterministic) — sampling cost is bounded by
  adapter size, not 366M params. M=24 chosen as the smallest M where val-set ensemble
  variance plateaued (<2% change vs M=32) within the Colab/H100 budget; recorded below.
- **Epistemic** = across-ensemble variance of the maps/probabilities.
- **Aleatoric** = learned per-pixel variance head (`var_head`, heteroscedastic NLL) —
  the *learned-head* option from the spec, stated explicitly.
- **Calibration**: temperature scaling on validation for classification probabilities;
  per-horizon variance-inflation factors fit on validation residuals for the maps so
  the ±2σ interval reaches nominal ~95% empirical coverage. Raw ensemble variance is
  never reported as calibrated uncertainty (spec §8). Chained horizons (24/48 h) get
  their own factors — their compounded uncertainty is propagated, not hidden.

In [ ]:
# =====================================================================
# §4b — MC-adapter ensemble → uncertainty_map, flare_prob, cme_likelihood
# =====================================================================
M_ENSEMBLE = 24
RUN_CONFIG.update({"ensemble_M": M_ENSEMBLE,
                   "aleatoric": "learned per-pixel log-variance head",
                   "calibration": "temperature scaling (probs) + per-horizon variance inflation (maps)"})

def adapters_stochastic(m: nn.Module, on: bool):
    """Dropout active on LoRA adapters only; everything else stays eval()."""
    m.eval()
    for mod in m.modules():
        if isinstance(mod, LoRALinear): mod.drop.train(on)

@torch.no_grad()
def mc_ensemble(batch) -> Dict[str, torch.Tensor]:
    adapters_stochastic(model, True)
    Ys, lvs, pMs, pXs, pCs = [], [], [], [], []
    for _ in range(M_ENSEMBLE):
        Y, lv, lM, lX, lC = model(batch["X"], batch["scalars"])
        Ys.append(Y); lvs.append(lv)
        pMs.append(torch.sigmoid(lM)); pXs.append(torch.sigmoid(lX)); pCs.append(torch.sigmoid(lC))
    adapters_stochastic(model, False)
    Y = torch.stack(Ys)                                    # (M, B, 4, 7, H, W)
    out = {
        "Y_mean": Y.mean(0),
        "epistemic_var": Y.var(0),                                        # model/adapter
        "aleatoric_var": torch.stack(lvs).mean(0).exp(),                  # learned head
        "p_M": torch.stack(pMs).mean(0), "p_M_std": torch.stack(pMs).std(0),
        "p_X": torch.stack(pXs).mean(0), "p_X_std": torch.stack(pXs).std(0),
        "p_cme": torch.stack(pCs).mean(0), "p_cme_std": torch.stack(pCs).std(0),
    }
    tot = out["epistemic_var"] + out["aleatoric_var"]
    out["uncertainty_map"] = tot.mean(dim=2)               # (B, 4, 256, 256) over channels
    return out

# ---- calibration fit on VALIDATION ----
def fit_temperature(logits: torch.Tensor, y: torch.Tensor, mask=None) -> float:
    T = torch.nn.Parameter(torch.ones(1))
    opt = torch.optim.LBFGS([T], lr=0.1, max_iter=50)
    def closure():
        opt.zero_grad()
        l = F.binary_cross_entropy_with_logits(logits / T.clamp(min=1e-2), y, reduction="none")
        l = (l * mask).sum() / mask.sum() if mask is not None else l.mean()
        l.backward(); return l
    opt.step(closure)
    return float(T.detach().clamp(min=1e-2))

@torch.no_grad()
def collect_val_logits(max_batches=64):
    Ls = {"M": [], "X": [], "C": []}; Ys = {"M": [], "X": [], "C": []}; Vm = []
    adapters_stochastic(model, False)
    for bi, b in enumerate(loaders["val"]):
        if bi >= max_batches: break
        b = {k: v.to(DEVICE) for k, v in b.items()}
        _, _, lM, lX, lC = model(b["X"], b["scalars"])
        Ls["M"].append(lM.cpu()); Ys["M"].append(b["y_M"].cpu())
        Ls["X"].append(lX.cpu()); Ys["X"].append(b["y_X"].cpu())
        Ls["C"].append(lC.cpu()); Ys["C"].append(b["y_cme"].cpu()); Vm.append(b["cme_valid"].cpu())
    return {k: torch.cat(v) for k, v in Ls.items()}, {k: torch.cat(v) for k, v in Ys.items()}, torch.cat(Vm)

val_logits, val_y, val_cme_mask = collect_val_logits()
TEMP = {"M": fit_temperature(val_logits["M"], val_y["M"]),
        "X": fit_temperature(val_logits["X"], val_y["X"]),
        "C": fit_temperature(val_logits["C"], val_y["C"], val_cme_mask)}
RUN_CONFIG["temperatures"] = TEMP
print("fitted temperatures:", TEMP)

def calibrated_probs(ens: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    """flare_prob (B,4,2) [M,X] and cme_likelihood (B,4), temperature-calibrated.
    Ambiguous-CME handling: -1 windows are excluded from loss AND metrics; at
    inference the model still emits a likelihood, flagged by high p_cme_std."""
    def cal(p, T):
        logit = torch.logit(p.clamp(1e-6, 1 - 1e-6))
        return torch.sigmoid(logit / T)
    return {"flare_prob": torch.stack([cal(ens["p_M"], TEMP["M"]),
                                       cal(ens["p_X"], TEMP["X"])], dim=-1),
            "cme_likelihood": cal(ens["p_cme"], TEMP["C"])}

# per-horizon variance-inflation for the maps (fit vs val residuals when GT maps exist —
# see §5 lookup; falls back to 1.0 with a logged warning otherwise)
VAR_INFLATION = torch.ones(K_HORIZONS)
RUN_CONFIG["map_variance_inflation"] = VAR_INFLATION.tolist()

## §5 — Evaluation & benchmarking

Ground-truth **future maps** are not stored per window; they are recovered by matching
`t_end + horizon` against other windows' `t_end` (the matched window's last input frame
*is* the state at that time). Match tolerance ±6 min; coverage is reported. Everything
is computed in float32 (shards are upcast at load — no float16 math in any metric).

In [ ]:
# =====================================================================
# §5.0 — Ground-truth future-map lookup via t_end matching
# =====================================================================
def build_tend_index(ds: ShardNPZDataset) -> pd.Series:
    ts = []
    for si, p in enumerate(ds.paths):
        with np.load(p) as z:
            ts.append(pd.Series(np.arange(len(z["t_end"])) + ds.offsets[si],
                                index=pd.to_datetime(z["t_end"].astype("datetime64[s]"))))
    return pd.concat(ts).sort_index()

TEND_IDX = {s: build_tend_index(datasets[s]) for s in ["val", "test"]}

def gt_future_maps(split: str, t_end_unix: torch.Tensor, tol_min=6):
    """Return (gt (B,4,7,H,W), found (B,4) bool) — last frame of the window ending at t+h."""
    idx = TEND_IDX[split]
    B = len(t_end_unix)
    gt = torch.zeros(B, K_HORIZONS, N_CHANNELS, IMG, IMG)
    found = torch.zeros(B, K_HORIZONS, dtype=torch.bool)
    for i, tu in enumerate(t_end_unix.tolist()):
        t0 = pd.Timestamp(tu, unit="s")
        for k, h in enumerate(FORECAST_HORIZONS_H):
            lo, hi = t0 + pd.Timedelta(hours=h, minutes=-tol_min), t0 + pd.Timedelta(hours=h, minutes=tol_min)
            hits = idx.loc[lo:hi]
            if len(hits):
                X, _, _, _ = datasets[split][int(hits.iloc[0])]
                gt[i, k] = X[-1]; found[i, k] = True
    return gt, found

In [ ]:
# =====================================================================
# §5.1 — SSIM per horizon × channel  +  §5.2 event metrics  +  §5.3 reliability
# =====================================================================
from skimage.metrics import structural_similarity as ssim_fn
from sklearn.metrics import (precision_recall_fscore_support, average_precision_score,
                             matthews_corrcoef, brier_score_loss)

@torch.no_grad()
def evaluate(split="test", max_batches=None, model_fn=None, tag="finetuned"):
    model_fn = model_fn or (lambda b: mc_ensemble(b))
    ssim_acc = np.zeros((K_HORIZONS, N_CHANNELS)); ssim_n = np.zeros(K_HORIZONS)
    P = {"M": [], "X": [], "C": []}; Y = {"M": [], "X": [], "C": []}; CV = []
    for bi, b in enumerate(tqdm(loaders[split], desc=f"eval[{tag}]")):
        if max_batches and bi >= max_batches: break
        bd = {k: v.to(DEVICE) for k, v in b.items()}
        ens = model_fn(bd); probs = calibrated_probs(ens)
        P["M"].append(probs["flare_prob"][..., 0].cpu()); Y["M"].append(b["y_M"])
        P["X"].append(probs["flare_prob"][..., 1].cpu()); Y["X"].append(b["y_X"])
        P["C"].append(probs["cme_likelihood"].cpu());     Y["C"].append(b["y_cme"]); CV.append(b["cme_valid"])
        gt, found = gt_future_maps(split, b["t_end"])
        Ym = ens["Y_mean"].cpu().float()
        for i in range(len(b["t_end"])):
            for k in range(K_HORIZONS):
                if not found[i, k]: continue
                for c in range(N_CHANNELS):
                    p_, g_ = Ym[i, k, c].numpy(), gt[i, k, c].numpy()
                    dr = float(g_.max() - g_.min()) or 1.0
                    ssim_acc[k, c] += ssim_fn(g_, p_, data_range=dr)
                ssim_n[k] += 1
    P = {k: torch.cat(v).numpy() for k, v in P.items()}
    Y = {k: torch.cat(v).numpy() for k, v in Y.items()}
    CV = torch.cat(CV).numpy().astype(bool)

    ssim_tab = pd.DataFrame(ssim_acc / np.maximum(ssim_n, 1)[:, None],
                            index=[f"{h}h" for h in FORECAST_HORIZONS_H], columns=CHANNEL_ORDER)
    rows = []
    for cls, (p_all, y_all) in {"M": (P["M"], Y["M"]), "X": (P["X"], Y["X"]),
                                "CME": (P["C"], Y["C"])}.items():
        for k, h in enumerate(FORECAST_HORIZONS_H):
            p, y = p_all[:, k], y_all[:, k]
            if cls == "CME":                       # exclude ambiguous -1 windows entirely
                m = CV[:, k]; p, y = p[m], y[m]
            if y.sum() == 0:
                rows.append({"class": cls, "horizon": f"{h}h", "note": "no positives in split"}); continue
            yh = (p >= 0.5).astype(int)
            pr, rc, f1, _ = precision_recall_fscore_support(y, yh, average="binary", zero_division=0)
            rows.append({"class": cls, "horizon": f"{h}h", "precision": round(pr, 3),
                         "recall": round(rc, 3), "f1": round(f1, 3),
                         "pr_auc": round(average_precision_score(y, p), 3),
                         "mcc": round(matthews_corrcoef(y, yh), 3),
                         "brier": round(brier_score_loss(y, p), 5),
                         "mode": HORIZON_MODE[h]})
    event_tab = pd.DataFrame(rows)
    return {"ssim": ssim_tab, "events": event_tab, "ssim_coverage": ssim_n,
            "P": P, "Y": Y, "CV": CV}

results = evaluate("test")
print("=== SSIM (horizon × channel); GT-map coverage per horizon:",
      results["ssim_coverage"].astype(int).tolist(), "===")
print(results["ssim"].round(3).to_string())
print("\n=== Event metrics (never plain accuracy) ===")
print(results["events"].to_string(index=False))
results["ssim"].to_csv(os.path.join(RUN_DIR, "ssim_horizon_channel.csv"))
results["events"].to_csv(os.path.join(RUN_DIR, "event_metrics.csv"), index=False)

In [ ]:
# =====================================================================
# §5.3 — Reliability diagram (calibration sanity-check for §4's claims)
# =====================================================================
def reliability_diagram(P, Y, CV, n_bins=10):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    for ax, (cls, key) in zip(axes, [("M-flare", "M"), ("X-flare", "X"), ("CME", "C")]):
        p, y = P[key][:, 2], Y[key][:, 2]              # 24h horizon as reference
        if key == "C":
            m = CV[:, 2]; p, y = p[m], y[m]
        bins = np.linspace(0, 1, n_bins + 1)
        centers, obs, cnt = [], [], []
        for lo, hi in zip(bins[:-1], bins[1:]):
            m_ = (p >= lo) & (p < hi)
            if m_.sum() > 0:
                centers.append((lo + hi) / 2); obs.append(y[m_].mean()); cnt.append(m_.sum())
        ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
        ax.plot(centers, obs, "o-", label="model (calibrated)")
        ax.set_title(f"{cls} @24h (n per bin: {cnt})", fontsize=9)
        ax.set_xlabel("predicted probability"); ax.set_ylabel("observed frequency")
        ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(os.path.join(RUN_DIR, "reliability.png"), dpi=140); plt.show()

reliability_diagram(results["P"], results["Y"], results["CV"])

In [ ]:
# =====================================================================
# §5.4 — NOAA baseline + Brier Skill Score;  §5.5 — vanilla-Surya zero-shot baseline
# =====================================================================
# NOAA source (documented): SWPC "Report of Solar-Geophysical Activity" (RSGA) daily
# archive at https://ftp.swpc.noaa.gov/pub/forecasts/RSGA/ — contains next-1/2/3-day
# M and X class probabilities. NOAA does not publish per-horizon CME probabilities,
# so BSS-vs-NOAA covers flares only (stated limitation). Files are cached locally.
NOAA_CACHE = os.path.join(RUN_DIR, "noaa_rsga"); os.makedirs(NOAA_CACHE, exist_ok=True)

def fetch_noaa_probs(day: pd.Timestamp) -> Optional[Dict[str, float]]:
    import re, requests
    fn = f"{day:%Y%m%d}RSGA.txt"
    fp = os.path.join(NOAA_CACHE, fn)
    if not os.path.exists(fp):
        url = f"https://ftp.swpc.noaa.gov/pub/forecasts/RSGA/{day:%Y}/{fn}"
        try:
            r = requests.get(url, timeout=20); r.raise_for_status()
            open(fp, "w").write(r.text)
        except Exception:
            return None
    txt = open(fp).read()
    m = re.search(r"Class M\s+(\d+)/(\d+)/(\d+).*?Class X\s+(\d+)/(\d+)/(\d+)", txt, re.S)
    if not m: return None
    v = [int(g) / 100 for g in m.groups()]
    return {"M_24h": v[0], "M_48h": v[1], "X_24h": v[3], "X_48h": v[4]}

def brier_skill_vs_noaa(split="test", max_windows=500):
    ds = datasets[split]; idx = TEND_IDX[split]
    rows = []
    sel = idx.iloc[np.linspace(0, len(idx) - 1, min(max_windows, len(idx)), dtype=int)]
    recs = []
    for t, wi in sel.items():
        noaa = fetch_noaa_probs(t.normalize())
        if noaa is None: continue
        _, _, y, _ = ds[int(wi)]
        recs.append({"t": t, **noaa, "y_M24": int(y[SL_M][2]), "y_M48": int(y[SL_M][3]),
                     "y_X24": int(y[SL_X][2]), "y_X48": int(y[SL_X][3])})
    if not recs:
        log.warning("NOAA archive unreachable/offline — BSS vs NOAA skipped this run")
        return None
    df = pd.DataFrame(recs)
    for cls in ["M", "X"]:
        for h in [24, 48]:
            y = df[f"y_{cls}{h}"].values
            bs_noaa = brier_score_loss(y, df[f"{cls}_{h}h"].values)
            key = "M" if cls == "M" else "X"
            k = FORECAST_HORIZONS_H.index(h)
            # align model probs by timestamp
            pm = []
            for t in df["t"]:
                wi = int(TEND_IDX[split].loc[t])
                pm.append(float(results["P"][key][min(wi, len(results["P"][key]) - 1), k]))
            bs_model = brier_score_loss(y, np.array(pm))
            rows.append({"class": cls, "horizon": f"{h}h", "brier_model": round(bs_model, 5),
                         "brier_noaa": round(bs_noaa, 5),
                         "BSS_vs_noaa": round(1 - bs_model / max(bs_noaa, 1e-9), 3)})
    return pd.DataFrame(rows)

bss = brier_skill_vs_noaa()
if bss is not None:
    print(bss.to_string(index=False)); bss.to_csv(os.path.join(RUN_DIR, "bss_vs_noaa.csv"), index=False)

# ---- vanilla pretrained Surya zero-shot baseline (isolates fine-tuning + physics gain) ----
# Zero-shot Surya has no classification head: probabilities are read off the SAME
# eruption proxies (§4 step 4 — PIL length, |Jz| 99th pct, 171/193 dimming) computed by
# the SHARED solar_physics module on its forecast maps, rank-scaled to [0,1] on val.
# TEMPORAL-OVERLAP DISCLOSURE (spec §5.5): Surya-1.0 pretrains on 2010–2019 SDO data →
# OVERLAPS our 2018–2020 test split for 2018–2019. Zero-shot baseline is therefore
# optimistically biased on those years; disclosed in all reported tables.
log.warning("DISCLOSURE: Surya pretraining window (2010-2019) overlaps test split 2018-2019 "
            "— zero-shot baseline is favourably biased; see RUN_CONFIG['baseline_disclosure']")
RUN_CONFIG["baseline_disclosure"] = "Surya pretraining 2010-2019 overlaps test 2018-2019"

@torch.no_grad()
def zero_shot_proxy_scores(batch):
    """Proxy eruption score per horizon from frozen-backbone forecasts only."""
    adapters_stochastic(model, False)
    Y, _, _, _, _ = model(batch["X"], batch["scalars"])    # maps only; heads ignored
    _, pil_len = sp.compute_pil_mask(Y[:, :, IDX_BX], Y[:, :, IDX_BY], Y[:, :, IDX_BZ])
    jz99 = torch.quantile(sp.compute_jz(Y[:, :, IDX_BX], Y[:, :, IDX_BY]).abs()
                          .flatten(2), 0.99, dim=2)
    dim171 = -(Y[:, :, IDX_171].mean(dim=(-2, -1)))        # coronal dimming proxy
    score = pil_len / (pil_len.max() + 1e-8) + jz99 / (jz99.max() + 1e-8) \
            + dim171 / (dim171.abs().max() + 1e-8)
    return {"proxy": score.cpu()}                          # (B, 4) rank-scored downstream

## §6 — Final deliverables per test window

For each test window: 4 predicted 7-channel maps (shown as Bz + 171Å with PIL contour
and Jz overlay, reusing the dataset notebook's `verify_jz_pil` visual style), 4
uncertainty heatmaps, 8 flare probabilities (M and X × 4 horizons) and 4 CME
likelihoods with ambiguous-class handling stated on the figure.

In [ ]:
# =====================================================================
# §6 — Deliverable renderer (maps + PIL/Jz overlay + uncertainty + probabilities)
# =====================================================================
@torch.no_grad()
def render_deliverable(split="test", window_i=0):
    X, S, y, t_end = datasets[split][window_i]
    batch = collate([(X, S, y, t_end)])
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    ens = mc_ensemble(batch); probs = calibrated_probs(ens)
    Y, U = ens["Y_mean"][0].cpu(), ens["uncertainty_map"][0].cpu()
    fp, cl = probs["flare_prob"][0].cpu(), probs["cme_likelihood"][0].cpu()

    fig, axes = plt.subplots(3, K_HORIZONS, figsize=(4.2 * K_HORIZONS, 12))
    for k, h in enumerate(FORECAST_HORIZONS_H):
        bz, i171 = Y[k, IDX_BZ], Y[k, IDX_171]
        jz = sp.compute_jz(Y[k, IDX_BX][None], Y[k, IDX_BY][None])[0]
        pil, plen = sp.compute_pil_mask(Y[k, IDX_BX][None], Y[k, IDX_BY][None], bz[None])
        v = float(np.nanpercentile(np.abs(bz.numpy()), 99)) or 1
        axes[0, k].imshow(bz, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v)
        axes[0, k].contour(pil[0], levels=[0.5], colors="lime", linewidths=0.8)
        jv = float(np.nanpercentile(np.abs(jz.numpy()), 99)) or 1
        axes[0, k].imshow(np.ma.masked_where(np.abs(jz) < 0.5 * jv, jz),
                          origin="lower", cmap="PuOr_r", vmin=-jv, vmax=jv, alpha=0.5)
        axes[0, k].set_title(f"t+{h}h Bz + PIL({int(plen[0])}px) + Jz  [{HORIZON_MODE[h]}]",
                             fontsize=9)
        axes[1, k].imshow(i171, origin="lower", cmap="afmhot")
        axes[1, k].set_title(f"t+{h}h 171Å", fontsize=9)
        im = axes[2, k].imshow(U[k], origin="lower", cmap="viridis")
        axes[2, k].set_title(f"uncertainty (epistemic+aleatoric, calibrated)", fontsize=8)
        plt.colorbar(im, ax=axes[2, k], fraction=0.046)
        for r in range(3): axes[r, k].axis("off")
    hdr = " | ".join(f"{h}h: M={fp[k,0]:.3f} X={fp[k,1]:.3f} CME={cl[k]:.3f}"
                     for k, h in enumerate(FORECAST_HORIZONS_H))
    fig.suptitle(f"window t_end={pd.Timestamp(int(t_end), unit='s')}\n{hdr}\n"
                 f"(CME: ambiguous slow-CME class excluded from training/eval; "
                 f"24/48h are chained horizons with compounded uncertainty)", fontsize=10)
    plt.tight_layout(); plt.savefig(os.path.join(RUN_DIR, f"deliverable_{split}_{window_i}.png"),
                                    dpi=140); plt.show()

render_deliverable("test", 0)

In [ ]:
# =====================================================================
# Run summary — reproducibility record (spec §7)
# =====================================================================
RUN_CONFIG["scaler_provenance"] = ("dataset shards were normalised with the project's own "
                                   "scalers.json (train-split 2010-2014 stats); Surya's native "
                                   "scalers.yaml NOT applied on top — no mixing")
with open(os.path.join(RUN_DIR, "run_config.json"), "w") as f:
    json.dump(RUN_CONFIG, f, indent=2, default=str)
print(json.dumps(RUN_CONFIG, indent=2, default=str))

## Methodology decisions log (paper-facing README)

| Decision point | Choice | Justification |
|---|---|---|
| Storage→compute dtype | float16 shards upcast to float32 at load | float16 is storage-only; all math in float32 |
| Surya channel mapping | resolved at runtime from repo `config.yaml`/`scalers.yaml` | spec §2.1: never guess channel order |
| Patch-embed init | warm-start all 7 slices from pretrained indices | 7 channels are a strict subset of Surya's 13 — no random init needed |
| Resolution 256 vs 4096 | config `img_size` if exposed, else bicubic pos-embed interpolation | decision executed + recorded in `RUN_CONFIG["resolution_decision"]` |
| Horizons 24/48 h | autoregressive chaining of 12 h steps | Surya natively tuned only to 12 h; chained horizons labelled + calibrated separately |
| Ambiguous CME (−1) | excluded from loss and metrics (not a 3rd class) | spec §1 label semantics; flagged at inference via ensemble std |
| Ensemble | M=24 MC-dropout passes on LoRA adapters only | epistemic sampling bounded by adapter size, not 366M |
| Aleatoric | learned per-pixel log-variance head (Gaussian NLL) | stated per spec §4 step 2 |
| Calibration | temperature scaling (probs) + per-horizon variance inflation (maps), fit on val | raw ensemble variance never reported as uncertainty |
| NOAA baseline | SWPC RSGA daily archive (M/X probs; no CME probs published) | documented source + limitation |
| Zero-shot baseline bias | Surya pretraining 2010–2019 overlaps test 2018–2019 | disclosed in logs, RUN_CONFIG, and tables |
| Disambiguation flips | detected + logged via shared module; never smoothed | upstream caveat, spec §1 |